# Model Evaluation

This notebook evaluates the trained coding agent model.

**Evaluation Metrics:**
1. Loss on test set
2. BLEU score for code generation quality
3. Exact match accuracy
4. Qualitative analysis

In [ ]:
import torch
import pandas as pd
import json
import numpy as np
from pathlib import Path
from transformers import T5ForConditionalGeneration, RobertaTokenizer
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set paths
PROCESSED_DATA_DIR = Path('../data/processed')
MODEL_DIR = Path('../models').resolve()

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Test Data and Model

In [ ]:
# Load test data
test_data = pd.read_json(PROCESSED_DATA_DIR / 'test.json')
print(f"Test samples: {len(test_data)}")

# Load trained model
model_path = (MODEL_DIR / 'best_model').resolve()
# Check if model exists first to avoid HFValidationError on Windows
if not model_path.exists():
    raise FileNotFoundError(f"Model not found at {model_path}. Please train the model first.")
model = T5ForConditionalGeneration.from_pretrained(model_path.as_posix())
tokenizer = RobertaTokenizer.from_pretrained(model_path.as_posix())
model = model.to(device)
model.eval()

print(f"Model loaded from: {model_path}")

# Load training config
with open(MODEL_DIR / 'training_config.json', 'r') as f:
    config = json.load(f)

print("\nTest data distribution:")
print(test_data['language'].value_counts())

## 2. Calculate Test Loss

In [ ]:
def calculate_test_loss(model, test_data, tokenizer, device, batch_size=4):
    """Calculate average loss on test set"""
    model.eval()
    total_loss = 0
    num_batches = 0
    
    with torch.no_grad():
        for i in tqdm(range(0, len(test_data), batch_size), desc="Calculating loss"):
            batch = test_data.iloc[i:i+batch_size]
            
            # Tokenize inputs
            inputs = tokenizer(
                batch['input'].tolist(),
                max_length=config['max_input_length'],
                padding=True,
                truncation=True,
                return_tensors='pt'
            ).to(device)
            
            # Tokenize outputs
            labels = tokenizer(
                batch['output'].tolist(),
                max_length=config['max_output_length'],
                padding=True,
                truncation=True,
                return_tensors='pt'
            )['input_ids'].to(device)
            
            # Replace padding with -100
            labels[labels == tokenizer.pad_token_id] = -100
            
            # Forward pass
            outputs = model(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                labels=labels
            )
            
            total_loss += outputs.loss.item()
            num_batches += 1
    
    return total_loss / num_batches

test_loss = calculate_test_loss(model, test_data, tokenizer, device)
perplexity = np.exp(test_loss)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Perplexity: {perplexity:.2f}")

## 3. Generate Predictions

In [ ]:
def generate_predictions(model, test_data, tokenizer, device, num_samples=None):
    """Generate predictions for test samples"""
    model.eval()
    predictions = []
    
    if num_samples is None:
        num_samples = len(test_data)
    
    with torch.no_grad():
        for i in tqdm(range(num_samples), desc="Generating predictions"):
            row = test_data.iloc[i]
            
            # Tokenize input
            inputs = tokenizer(
                row['input'],
                return_tensors='pt',
                max_length=config['max_input_length'],
                truncation=True
            ).to(device)
            
            # Generate
            outputs = model.generate(
                inputs['input_ids'],
                max_length=config['max_output_length'],
                num_beams=config['num_beams'],
                temperature=config['temperature'],
                early_stopping=True
            )
            
            # Decode
            predicted_code = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            predictions.append({
                'language': row['language'],
                'framework': row['framework'],
                'category': row['category'],
                'description': row['description'],
                'expected': row['output'],
                'predicted': predicted_code
            })
    
    return pd.DataFrame(predictions)

# Generate predictions for all test samples
predictions_df = generate_predictions(model, test_data, tokenizer, device)
print(f"\nGenerated {len(predictions_df)} predictions")

## 4. Calculate BLEU Score

In [ ]:
from sklearn.metrics import accuracy_score

def simple_bleu(reference, candidate):
    """Simple BLEU-like score based on token overlap"""
    ref_tokens = set(reference.split())
    cand_tokens = set(candidate.split())
    
    if len(cand_tokens) == 0:
        return 0.0
    
    overlap = len(ref_tokens.intersection(cand_tokens))
    precision = overlap / len(cand_tokens)
    recall = overlap / len(ref_tokens) if len(ref_tokens) > 0 else 0
    
    if precision + recall == 0:
        return 0.0
    
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1

# Calculate BLEU scores
predictions_df['bleu_score'] = predictions_df.apply(
    lambda row: simple_bleu(row['expected'], row['predicted']),
    axis=1
)

# Calculate exact match
predictions_df['exact_match'] = predictions_df.apply(
    lambda row: row['expected'].strip() == row['predicted'].strip(),
    axis=1
)

# Overall metrics
avg_bleu = predictions_df['bleu_score'].mean()
exact_match_rate = predictions_df['exact_match'].mean()

print(f"\nOverall Metrics:")
print(f"  Average BLEU Score: {avg_bleu:.4f}")
print(f"  Exact Match Rate: {exact_match_rate:.2%}")

# Metrics by language
print(f"\nMetrics by Language:")
for lang in predictions_df['language'].unique():
    lang_df = predictions_df[predictions_df['language'] == lang]
    print(f"  {lang}:")
    print(f"    BLEU: {lang_df['bleu_score'].mean():.4f}")
    print(f"    Exact Match: {lang_df['exact_match'].mean():.2%}")

## 5. Visualize Results

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# BLEU score distribution
axes[0, 0].hist(predictions_df['bleu_score'], bins=20, color='skyblue', edgecolor='black')
axes[0, 0].axvline(avg_bleu, color='red', linestyle='--', linewidth=2, label=f'Mean: {avg_bleu:.3f}')
axes[0, 0].set_xlabel('BLEU Score', fontsize=12)
axes[0, 0].set_ylabel('Frequency', fontsize=12)
axes[0, 0].set_title('BLEU Score Distribution', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# BLEU by language
bleu_by_lang = predictions_df.groupby('language')['bleu_score'].mean().sort_values()
bleu_by_lang.plot(kind='barh', ax=axes[0, 1], color='lightcoral')
axes[0, 1].set_xlabel('Average BLEU Score', fontsize=12)
axes[0, 1].set_title('BLEU Score by Language', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='x')

# Exact match by language
exact_by_lang = predictions_df.groupby('language')['exact_match'].mean().sort_values()
exact_by_lang.plot(kind='barh', ax=axes[1, 0], color='lightgreen')
axes[1, 0].set_xlabel('Exact Match Rate', fontsize=12)
axes[1, 0].set_title('Exact Match Rate by Language', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='x')

# BLEU by category
bleu_by_cat = predictions_df.groupby('category')['bleu_score'].mean().sort_values(ascending=False)
bleu_by_cat.plot(kind='bar', ax=axes[1, 1], color='plum')
axes[1, 1].set_xlabel('Category', fontsize=12)
axes[1, 1].set_ylabel('Average BLEU Score', fontsize=12)
axes[1, 1].set_title('BLEU Score by Category', fontsize=14, fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(MODEL_DIR / 'evaluation_results.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nEvaluation results saved to: {MODEL_DIR / 'evaluation_results.png'}")

## 6. Qualitative Analysis - Sample Predictions

In [ ]:
# Show best and worst predictions
print("="*80)
print("TOP 3 PREDICTIONS (Highest BLEU Score)")
print("="*80)

top_predictions = predictions_df.nlargest(3, 'bleu_score')
for i, (idx, row) in enumerate(top_predictions.iterrows(), 1):
    print(f"\n{i}. Language: {row['language']} | Category: {row['category']} | BLEU: {row['bleu_score']:.3f}")
    print(f"Description: {row['description']}")
    print(f"\nExpected:\n{row['expected'][:200]}...")
    print(f"\nPredicted:\n{row['predicted'][:200]}...")
    print("-" * 80)

print("\n" + "="*80)
print("BOTTOM 3 PREDICTIONS (Lowest BLEU Score)")
print("="*80)

bottom_predictions = predictions_df.nsmallest(3, 'bleu_score')
for i, (idx, row) in enumerate(bottom_predictions.iterrows(), 1):
    print(f"\n{i}. Language: {row['language']} | Category: {row['category']} | BLEU: {row['bleu_score']:.3f}")
    print(f"Description: {row['description']}")
    print(f"\nExpected:\n{row['expected'][:200]}...")
    print(f"\nPredicted:\n{row['predicted'][:200]}...")
    print("-" * 80)

## 7. Save Evaluation Results

In [ ]:
# Save detailed predictions
predictions_df.to_json(
    MODEL_DIR / 'test_predictions.json',
    orient='records',
    indent=2
)

# Save evaluation summary
eval_summary = {
    'test_loss': float(test_loss),
    'perplexity': float(perplexity),
    'avg_bleu_score': float(avg_bleu),
    'exact_match_rate': float(exact_match_rate),
    'num_test_samples': len(test_data),
    'metrics_by_language': {
        lang: {
            'bleu': float(predictions_df[predictions_df['language'] == lang]['bleu_score'].mean()),
            'exact_match': float(predictions_df[predictions_df['language'] == lang]['exact_match'].mean())
        }
        for lang in predictions_df['language'].unique()
    }
}

with open(MODEL_DIR / 'evaluation_summary.json', 'w') as f:
    json.dump(eval_summary, f, indent=2)

print(f"Predictions saved to: {MODEL_DIR / 'test_predictions.json'}")
print(f"Evaluation summary saved to: {MODEL_DIR / 'evaluation_summary.json'}")

print("\n" + "="*80)
print("EVALUATION COMPLETE!")
print("="*80)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Perplexity: {perplexity:.2f}")
print(f"Average BLEU Score: {avg_bleu:.4f}")
print(f"Exact Match Rate: {exact_match_rate:.2%}")
print(f"\nNext step: Run notebook 06_inference_demo.ipynb to test the model interactively")